# Modular semi-Mapper pipeline — CTN-0051

This notebook is a thin driver over the `mapper` package. All logic lives in the package modules; here you only **tune parameters and call the pipeline**.

Pipeline: `data -> distance -> lens -> cover -> graph -> layout -> viz`.

Three lenses are available:
- **feature**    — a data column (original behaviour, e.g. `attendance_density_w8`)
- **centrality** — graph centrality on the proximity graph (degree, betweenness, ...)
- **density**    — local density in feature space (knn, ball, kde)

## 1. Parameters — tune here

In [34]:
from mapper import MapperParams, run_pipeline, visualise
from mapper import diagnostics as dg
import numpy as np

params = MapperParams(
    PRE_TRIAL_CSV="../../data/clean-data/pre-trial.csv",
    TARGET_CSV   ="../../data/clean-data/retention_tier.csv",
    W8_CSV       ="../../data/clean-data/features_w8.csv",
    W12_CSV      ="../../data/clean-data/features_w12.csv",
    W24_CSV      ="../../data/clean-data/features_w24.csv",

    # --- proximity ---
    EPSILON=0.6
    ,
    METRIC ="cosine",        # euclidean | cosine | manhattan | minkowski
    MINKOWSKI_P= np.inf,            # only for minkowski

    # --- LENS: pick one of the three ---
    LENS_KIND="feature",        # "feature" | "centrality" | "density"
    FEATURE_LENS_COL  = "attendance_density_w24",   # feature lens: attendance_density_w8 | detox_los 
    CENTRALITY_MEASURE="betweenness",             # centrality lens
    DENSITY_METHOD    ="knn",                     # density lens
    DENSITY_K         =10,

    # --- COVER / BINNING (tunable) ---
    COVER_MODE ="piecewise",      # "uniform" (N_INTERVALS+OVERLAP) or "edges" (BIN_EDGES)
    N_INTERVALS= 60,
    OVERLAP    =0.7,
    # For explicit clinical bins instead, use:
    # COVER_MODE="edges", BIN_EDGES=[0,3,7,14,21], BIN_LABELS=["0-3","3-7","7-14","14+"]

    PIECEWISE_SEGMENTS=[
        (0.0, 0.3, 15),    # dense cover in the first half
        (0.3, 0.6, 5),
        (0.6, 1.0, 10)   # sparse cover in the second half
    ],



    # --- edge rule for the displayed graph ---
    EDGE_RULE  ="cover",        # "cover" (share a set) | "gap" | "none"
    MAX_BIN_GAP=1,

    # --- layout & colour ---
    LAYOUT  ="spring", 
    SPRING_K = 1,              # spring | spectral | pca
    COLOR_BY="tier",            # "tier" or "lens"
)
print(params.summary())

ε=0.6 | metric=cosine | lens=feature=attendance_density_w24 | cover=edges | edge_rule=cover | layout=spring


## 2. Run the pipeline

In [35]:
result = run_pipeline(params)   # prints stage-by-stage summaries

Patients        : 554
Feature matrix  : (554, 58)
Missing values  : 0

Retention tier distribution:
retention_tier
1    164
2    108
3     61
4    221 

Distance matrix : (554, 554)
Distance range  : [0.000, 1.699]
Percentiles:
   10th : 0.700
   25th : 0.856
   50th : 1.013
   75th : 1.157
   90th : 1.275
With ε = 0.6:
  Edges before pruning : 7197
  Edge density         : 4.7% 

[feature] attendance_density_w24: min=0.048 median=0.476 max=1.000 (n_valid=554) 

Cover mode: piecewise  |  30 sets

  Set  0 [0,0.0667)      :  94 patients
  Set  1 [0.02,0.0867)   :  94 patients
  Set  2 [0.04,0.107)    : 128 patients
  Set  3 [0.06,0.127)    :  34 patients
  Set  4 [0.08,0.147)    :  75 patients
  Set  5 [0.1,0.167)     :  41 patients
  Set  6 [0.12,0.187)    :  41 patients
  Set  7 [0.14,0.207)    :  78 patients
  Set  8 [0.16,0.227)    :  37 patients
  Set  9 [0.18,0.247)    :  49 patients
  Set 10 [0.2,0.267)     :  12 patients
  Set 11 [0.22,0.287)    :  29 patients
  Set 12 [0.24,0.3

## 3. Interactive Bokeh graph

In [36]:
from bokeh.io import output_notebook
output_notebook()
visualise(result)

Loading BokehJS ...

figure(id='p1902', ...)

In [ ]:
from bokeh.io import output_file, save
import networkx as nx
import copy

# Get the figure without rendering it in the notebook
fig = visualise(result, render=False)

# Save Bokeh HTML
output_file("../../graphs/w12/att_density12_a.html", title="Mapper Graph – Week 12 Attendance Density")
save(fig)

# GraphML only supports scalar types — convert any list attributes to strings
G_export = copy.deepcopy(result.graph)
for _, data in G_export.nodes(data=True):
    for k, v in list(data.items()):
        if isinstance(v, list):
            data[k] = ",".join(map(str, v))
for _, _, data in G_export.edges(data=True):
    for k, v in list(data.items()):
        if isinstance(v, list):
            data[k] = ",".join(map(str, v))

nx.write_graphml(G_export, "../../graphs/w12/att_density12_a.graphml")
# Reload later:
G = nx.read_graphml("../../graphs/w12/att_density12_a.graphml")